# Week 3 — tools and a hand-written agent loop, in code

One short theory cell, then a standalone snippet per concept. Toy data (a shop and a fake incident), no dependency on `triage/`.  
Section numbers match the week-3 lessons. **[$]** = calls the paid API (fractions of a cent on Haiku 5.5).  
**[free]** = API call that is not billed (`count_tokens`, a 400). Unmarked = pure Python, runs offline.

Sections 4 and 5 run the loop against a **scripted fake client** first, so you can see every stopping path offline, then against the real API.

| § | Concepts |
|---|---|
| 0 | Setup, helpers |
| 1 | Tool definition (`name`, `description`, `input_schema`), `strict`, what tools cost in tokens |
| 2 | The round trip by hand: `tool_use` → your code → `tool_result`; the rules the API enforces |
| 3 | Parallel calls, `is_error` (and when *not* to set it), `tool_choice`, `disable_parallel_tool_use` |
| 4 | The loop: stop reasons, step cap, token budget, duplicate calls, graceful wrap-up, final structured report, cost growth |
| 5 | Hygiene: truncating results (and saying so), one-line trace, per-run transcript JSONL, transcript viewer |

## 0. Setup

In [ ]:
import json, time, uuid
from pathlib import Path
from types import SimpleNamespace as NS
from concurrent.futures import ThreadPoolExecutor
import anthropic
from pydantic import BaseModel, Field
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))                # ANTHROPIC_API_KEY from .env
client = anthropic.Anthropic(timeout=60.0, max_retries=4)
MODEL = "claude-haiku-5-5"
PRICE = {"claude-haiku-5-5": (0.10, 0.50), "claude-sonnet-5-5": (2.00, 10.00)}   # $/M input, output
print("SDK", anthropic.__version__)

In [ ]:
def text_of(msg) -> str:
    # Only text blocks. On current models content[0] is often a (possibly empty) thinking block.
    return "".join(b.text for b in msg.content if b.type == "text")

def tool_uses(msg) -> list:
    # The tool_use blocks of one response. There can be 0, 1 or several (parallel calls).
    return [b for b in msg.content if b.type == "tool_use"]

def show_blocks(msg):
    # One line per content block, so you see exactly what the model sent.
    print(f"stop_reason={msg.stop_reason}")
    for b in msg.content:
        if b.type == "tool_use":
            print(f"  tool_use  id={b.id} name={b.name} input={json.dumps(b.input)}")
        elif b.type == "text":
            print(f"  text      {b.text[:120]!r}")
        else:
            print(f"  {b.type}")                     # thinking: empty text by default, keep it anyway

## 1. What a tool is

A tool is a **contract in the request**: `name`, `description`, `input_schema`.  
The model never runs anything; it writes a `tool_use` block and stops. Your code runs the function.  
The model sees only the definition, so the **description is a prompt**: say *when* to call it, not just what it does.

In [ ]:
# The functions YOUR code owns. The model never sees this code.
ORDERS = {"1042": "shipped, arrives Nov 11", "1043": "processing"}
STOCK = {"MUG-BLU": 0, "MUG-RED": 7, "TEE-BLK": 12}

def get_order(order_id: str) -> str:
    return ORDERS.get(order_id, f"No order with id {order_id}.")      # not found = an answer, not an error

def check_stock(sku: str) -> str:
    if sku not in STOCK:                                              # bad input = an error with a fix
        raise ValueError(f"Unknown sku '{sku}'. Valid skus: {', '.join(STOCK)}")
    return f"{sku}: {STOCK[sku]} in stock"

FUNCS = {"get_order": get_order, "check_stock": check_stock}

In [ ]:
# The definitions the model sees. Descriptions say WHEN to call.
TOOLS = [
    {"name": "get_order",
     "description": "Get shipping status for one order. Call when the user mentions an order number.",
     "input_schema": {"type": "object",
                      "properties": {"order_id": {"type": "string", "description": "Digits only, e.g. 1042"}},
                      "required": ["order_id"]}},
    {"name": "check_stock",
     "description": "Get units in stock for one product sku. Call when the user asks about availability.",
     "input_schema": {"type": "object",
                      "properties": {"sku": {"type": "string", "description": "Product sku, e.g. MUG-RED"}},
                      "required": ["sku"]}},
]

# strict=True: arguments are constrained-decoded to the schema (week 1 mechanism, applied to tool input).
# Needs additionalProperties: false, and every property listed in required (use a null type for "optional").
def strict(tool: dict) -> dict:
    s = dict(tool["input_schema"], additionalProperties=False, required=list(tool["input_schema"]["properties"]))
    return {**tool, "strict": True, "input_schema": s}

STRICT_TOOLS = [strict(t) for t in TOOLS]
print(json.dumps(STRICT_TOOLS[1], indent=2))

```code
{
  "name": "check_stock",
  "description": "Get units in stock for one product sku. Call when the user asks about availability.",
  "input_schema": {
    "type": "object",
    "properties": {
      "sku": {
        "type": "string",
        "description": "Product sku, e.g. MUG-RED"
      }
    },
    "required": [
      "sku"
    ],
    "additionalProperties": false
  },
  "strict": true
}
```

### 1.1 What tools cost  [free]
Definitions are prompt tokens, resent on every request, first in the cache prefix (tools → system → messages).

In [ ]:
msgs = [{"role": "user", "content": "Hi"}]
without = client.messages.count_tokens(model=MODEL, messages=msgs).input_tokens
with_tools = client.messages.count_tokens(model=MODEL, messages=msgs, tools=TOOLS).input_tokens
print(f"no tools: {without}   with 2 tools: {with_tools}   tool overhead: {with_tools - without} tokens per request")
print(f"x 15 loop steps = {15 * (with_tools - without):,} input tokens for definitions alone (before caching)")

```code
no tools: 11   with 2 tools: 508   tool overhead: 497 tokens per request
x 15 loop steps = 7,455 input tokens for definitions alone (before caching)
```

## 2. The round trip by hand  [$]

Stateless API, so one tool call = **two requests**. Request 2 resends the history plus the assistant turn plus the `tool_result`.

In [ ]:
messages = [{"role": "user", "content": "Where is my order 1042?"}]
r1 = client.messages.create(model=MODEL, max_tokens=1024, tools=TOOLS, messages=messages)
show_blocks(r1)                                           # expect stop_reason=tool_use and one tool_use block

```code
stop_reason=tool_use
  tool_use  id=toolu_01XZZ7NgL6gVQqyrSp5v9LGF name=get_order input={"order_id": "1042"}
```

In [ ]:
call = tool_uses(r1)[0]
output = FUNCS[call.name](**call.input)                   # YOUR code touches the "external system"

messages.append({"role": "assistant", "content": r1.content})          # whole content, unchanged
messages.append({"role": "user", "content": [
    {"type": "tool_result", "tool_use_id": call.id, "content": output}  # id links result to request
]})
r2 = client.messages.create(model=MODEL, max_tokens=1024, tools=TOOLS, messages=messages)
show_blocks(r2)                                           # expect end_turn and a text answer
print("input tokens:", r1.usage.input_tokens, "->", r2.usage.input_tokens, "(history + tool_use + result)")

```code
stop_reason=end_turn
  text      'Your order 1042 has shipped and is scheduled to arrive on November 11.'
input tokens: 514 -> 581 (history + tool_use + result)
```

### 2.1 Rules the API enforces  [free: a 400 is not billed]
Every `tool_use` id needs a matching `tool_result` in the very next user message.

In [ ]:
broken = messages[:2] + [{"role": "user", "content": "never mind, just tell me"}]   # result missing
try:
    client.messages.create(model=MODEL, max_tokens=256, tools=TOOLS, messages=broken)
except anthropic.BadRequestError as e:
    print("400:", e.message[:200])

```code
400: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.2: `tool_use` ids were found without `tool_result` blocks immediately after: toolu_01XZZ7NgL6gVQqyrS
```

## 3. Parallel calls, `is_error`, `tool_choice`

- Independent needs → several `tool_use` blocks in **one** response. Run them, send **all** results in **one** user message.
- A failed tool → `tool_result` with `is_error: true` and an actionable message. Catch **per call**, never let it kill the loop, never drop the result (400).
- Zero matches is an answer, not an error.

In [ ]:
def run_one(block) -> dict:
    # Execute one tool_use block; always return a tool_result, never raise.
    try:
        fn = FUNCS.get(block.name)
        if fn is None:
            raise KeyError(f"No tool named {block.name}. Available: {', '.join(FUNCS)}")
        return {"type": "tool_result", "tool_use_id": block.id, "content": fn(**block.input)}
    except Exception as e:
        return {"type": "tool_result", "tool_use_id": block.id, "is_error": True,
                "content": f"{type(e).__name__}: {e}"[:500]}          # short, no traceback

def run_tools(msg) -> list:
    # Run all tool_use blocks of one response concurrently; results keep their ids.
    with ThreadPoolExecutor() as pool:
        return list(pool.map(run_one, tool_uses(msg)))

# Offline check with hand-made blocks: one success, one bad sku.
fake_calls = NS(content=[NS(type="tool_use", id="A", name="get_order", input={"order_id": "1042"}),
                         NS(type="tool_use", id="B", name="check_stock", input={"sku": "blue mug"})])
for r in run_tools(fake_calls):
    print(r)

```code
{'type': 'tool_result', 'tool_use_id': 'A', 'content': 'shipped, arrives Nov 11'}
{'type': 'tool_result', 'tool_use_id': 'B', 'is_error': True, 'content': "ValueError: Unknown sku 'blue mug'. Valid skus: MUG-BLU, MUG-RED, TEE-BLK"}
```

### 3.1 Parallel + error + recovery, live  [$]

In [ ]:
messages = [{"role": "user", "content": "Where is my order 1042, and is the blue mug in stock?"}]
for turn in range(1, 5):
    r = client.messages.create(model=MODEL, max_tokens=1024, tools=TOOLS, messages=messages)
    print(f"--- turn {turn}"); show_blocks(r)
    messages.append({"role": "assistant", "content": r.content})
    if r.stop_reason != "tool_use":
        break
    results = run_tools(r)
    for res in results:
        print("  ->", "ERROR" if res.get("is_error") else "ok", res["content"][:80])
    messages.append({"role": "user", "content": results})               # ALL results, ONE message

```code
--- turn 1
stop_reason=tool_use
  thinking
  tool_use  id=toolu_01M5t93fwTdUAZW51yY98Y5V name=get_order input={"order_id": "1042"}
  tool_use  id=toolu_01HtHhKTMqKPVtMBRgQhbMTX name=check_stock input={"sku": "MUG-BLUE"}
  -> ok shipped, arrives Nov 11
  -> ERROR ValueError: Unknown sku 'MUG-BLUE'. Valid skus: MUG-BLU, MUG-RED, TEE-BLK
--- turn 2
stop_reason=tool_use
  tool_use  id=toolu_019hEeaW55fVLuHispHyVjSD name=check_stock input={"sku": "MUG-BLU"}
  -> ok MUG-BLU: 0 in stock
--- turn 3
stop_reason=end_turn
  text      'Your order 1042 has shipped and is scheduled to arrive November 11.\n\nThe blue mug (SKU MUG-BLU) is currently out of stoc'
```

### 3.2 `tool_choice`  [$]

| value | meaning | Opus 5.5 / Sonnet 5.5 |
|---|---|---|
| `auto` | model decides (default) | ok |
| `none` | tools stay in the prompt, cannot be called | ok |
| `any` / `tool` | force a call | **400** |

Use `none` instead of deleting tools: deleting changes the start of the cache prefix. `disable_parallel_tool_use: true` = at most one call per response.

In [ ]:
q = [{"role": "user", "content": "Is MUG-RED in stock and where is order 1043?"}]

r = client.messages.create(model=MODEL, max_tokens=512, tools=TOOLS, messages=q, tool_choice={"type": "none"})
print("none ->"); show_blocks(r)                          # text only, no tool_use possible

r = client.messages.create(model=MODEL, max_tokens=512, tools=TOOLS, messages=q,
                           tool_choice={"type": "auto", "disable_parallel_tool_use": True})
print("auto + disable_parallel ->"); show_blocks(r)       # at most one tool_use

try:
    client.messages.create(model="claude-sonnet-5-5", max_tokens=256, tools=TOOLS, messages=q,
                           tool_choice={"type": "any"})
except anthropic.BadRequestError as e:
    print("forced on Sonnet 5.5 -> 400:", e.message[:160])

```code
none ->
stop_reason=end_turn
auto + disable_parallel ->
stop_reason=tool_use
  tool_use  id=toolu_01DEPHqZSBbymgThN8bA5dYS name=check_stock input={"sku": "MUG-RED"}
forced on Sonnet 5.5 -> 400: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'tool_choice: type "tool" and "any" are not supported for this model.'
```

## 4. The loop and every way it stops

| signal | from | action |
|---|---|---|
| `tool_use` | API | run tools, append, next step |
| `end_turn` | API | done → final report |
| `max_tokens` | API | output cut; a tool_use input may be half-written → never run it |
| `refusal` | API | stop, log |
| step cap | you | wrap up: "give best answer", `tool_choice: none` |
| token budget | you | same wrap-up |
| same call twice | you | return "already called, unchanged" |

### 4.1 A scripted fake client (offline)
Returns pre-written responses in order, so every stopping path can be shown without the API. Same attribute names as real SDK objects.

In [ ]:
def fake_resp(*blocks, stop="tool_use", inp=1500, out=80):
    return NS(content=list(blocks), stop_reason=stop, _request_id=f"req_fake_{uuid.uuid4().hex[:6]}",
              usage=NS(input_tokens=inp, output_tokens=out, cache_read_input_tokens=0,
                       cache_creation_input_tokens=0))

def tu(id, name, **inp): return NS(type="tool_use", id=id, name=name, input=inp)
def tx(text): return NS(type="text", text=text)

class FakeClient:
    # Mimics client.messages.create: pops the next scripted response; records each request's message count.
    def __init__(self, script): self.script, self.calls = list(script), []
    @property
    def messages(self): return self
    def create(self, **kw):
        self.calls.append(len(kw["messages"]))
        return self.script.pop(0)

### 4.2 The loop (the core of `agent.py`)

In [ ]:
class Run:
    # Everything one investigation needs to track. 'log' is set in section 5; here it is a no-op.
    def __init__(self, max_steps=15, max_input=200_000):
        self.max_steps, self.max_input = max_steps, max_input
        self.used_input, self.seen, self.steps = 0, {}, 0
        self.log = lambda event: None

def run_tools_dedup(msg, run: Run) -> list:
    # Like run_tools, but a repeated (name, input) gets the old result plus a note instead of re-running.
    results = []
    for b in tool_uses(msg):
        key = (b.name, json.dumps(b.input, sort_keys=True))
        if key in run.seen:
            r = {"type": "tool_result", "tool_use_id": b.id,
                 "content": "Already called with the same input; result unchanged:\n" + run.seen[key]}
        else:
            t0 = time.time(); r = run_one(b); run.seen[key] = r["content"]
            run.log({"type": "tool", "step": run.steps, "tool_use_id": b.id, "name": b.name, "input": b.input,
                     "is_error": r.get("is_error", False), "latency_s": round(time.time() - t0, 3),
                     "sent": r["content"]})
        results.append(r)
    return results

def agent_loop(cli, task: str, run: Run, system="You are a helpful shop assistant."):
    messages = [{"role": "user", "content": task}]
    while run.steps < run.max_steps:
        run.steps += 1
        resp = cli.messages.create(model=MODEL, max_tokens=4096, system=system, tools=TOOLS,
                                   messages=messages, cache_control={"type": "ephemeral"})
        u = resp.usage
        run.used_input += u.input_tokens + (u.cache_read_input_tokens or 0) + (u.cache_creation_input_tokens or 0)
        run.log({"type": "llm", "step": run.steps, "request_id": resp._request_id, "stop_reason": resp.stop_reason,
                 "usage": {"in": u.input_tokens, "out": u.output_tokens, "cache_read": u.cache_read_input_tokens or 0,
                           "cache_write": u.cache_creation_input_tokens or 0},
                 "tool_uses": [{"name": b.name, "input": b.input} for b in tool_uses(resp)]})
        messages.append({"role": "assistant", "content": resp.content})

        if resp.stop_reason == "end_turn":
            return messages, "end_turn"
        if resp.stop_reason in ("max_tokens", "refusal"):
            return messages, resp.stop_reason                      # do NOT run a possibly cut tool_use

        messages.append({"role": "user", "content": run_tools_dedup(resp, run)})
        if run.used_input > run.max_input:
            return wrap_up(cli, messages, run, "budget", system)
    return wrap_up(cli, messages, run, "step_cap", system)

def wrap_up(cli, messages, run, outcome, system):
    # Graceful stop: text AFTER the tool_results in the same user message, tools disabled for one last call.
    messages[-1]["content"].append({"type": "text", "text":
        f"Stop investigating ({outcome}). Answer now from the evidence so far and list what is unverified."})
    resp = cli.messages.create(model=MODEL, max_tokens=4096, system=system, tools=TOOLS,
                               messages=messages, tool_choice={"type": "none"})
    messages.append({"role": "assistant", "content": resp.content})
    return messages, outcome

### 4.3 Three offline runs: clean finish, a model stuck repeating itself, and a cut-off output

In [ ]:
# A) error -> recovery -> end_turn
fc = FakeClient([
    fake_resp(tu("A", "get_order", order_id="1042"), tu("B", "check_stock", sku="blue mug")),
    fake_resp(tu("C", "check_stock", sku="MUG-BLU")),
    fake_resp(tx("Order 1042 has shipped; the blue mug is out of stock."), stop="end_turn"),
])
msgs, outcome = agent_loop(fc, "Order 1042 and blue mug?", Run())
print("A:", outcome, "| messages sent per request:", fc.calls)

# B) model asks for the same thing forever -> dedup notes -> step cap -> wrap-up answer
stuck = [fake_resp(tu(f"S{i}", "check_stock", sku="MUG-RED")) for i in range(4)]
fc = FakeClient(stuck + [fake_resp(tx("MUG-RED has 7 in stock."), stop="end_turn")])
msgs, outcome = agent_loop(fc, "Is MUG-RED in stock?", Run(max_steps=4))
print("B:", outcome, "| last user msg ends with:", msgs[-2]["content"][-1]["text"][:60])

# C) output cut off mid tool call -> nothing is executed
fc = FakeClient([fake_resp(tu("X", "check_stock", sku="MUG-"), stop="max_tokens")])
msgs, outcome = agent_loop(fc, "Is MUG-RED in stock?", Run())
print("C:", outcome, "| tool executed?", any(m["role"] == "user" and isinstance(m["content"], list) for m in msgs))

```code
A: end_turn | messages sent per request: [1, 3, 5]
B: step_cap | last user msg ends with: Stop investigating (step_cap). Answer now from the evidence 
C: max_tokens | tool executed? False
```

### 4.4 Real run  [$]

In [ ]:
run = Run(max_steps=8)
msgs, outcome = agent_loop(client, "Where are orders 1042 and 1043, and do you have a black tee?", run)
print(outcome, "| steps:", run.steps, "| total input tokens:", run.used_input)
print(text_of(NS(content=msgs[-1]["content"])))

```code
end_turn | steps: 3 | total input tokens: 2475
Here's what I found:

- **Order 1042:** Shipped and arriving November 11.
- **Order 1043:** Still processing, so it hasn't shipped yet.
- **Black tee:** Yes, we have 12 in stock.
```

### 4.5 Final structured report  [$]
No forced tool needed: after the loop, one `parse()` call over the transcript.

In [ ]:
class Order(BaseModel):
    order_id: str
    status: str

class Stock(BaseModel):
    sku: str
    units: int

class ShopAnswer(BaseModel):          # lists of objects, not dicts: structured outputs needs fixed keys
    orders: list[Order]
    stock: list[Stock]
    outcome: str = Field(description="why the loop stopped: end_turn, step_cap, budget")

final = client.messages.parse(
    model=MODEL, max_tokens=2048, tools=TOOLS, tool_choice={"type": "none"},
    messages=msgs + [{"role": "user", "content": f"Return the final answer as JSON. outcome={outcome}"}],
    output_format=ShopAnswer)
print(final.parsed_output)

```code
orders=[Order(order_id='1042', status='shipped, arrives Nov 11'), Order(order_id='1043', status='processing')] stock=[Stock(sku='TEE-BLK', units=12)] outcome='end_turn'
```

### 4.6 Why cost grows with steps²
Every step resends everything before it.

In [ ]:
def loop_cost(steps, prefix=1550, per_step=1580, model="claude-sonnet-5-5", cached=False):
    pin = PRICE[model][0] / 1e6
    sent = [prefix + (k - 1) * per_step for k in range(1, steps + 1)]
    if not cached:
        return sum(sent), sum(sent) * pin
    new = [prefix] + [per_step] * (steps - 1)                 # written once at 1.25x
    old = [s - n for s, n in zip(sent, new)]                   # re-read at 0.1x
    eq = 1.25 * sum(new) + 0.1 * sum(old)
    return sum(sent), eq * pin

print(f"{'steps':>5} {'tokens sent':>12} {'$ uncached':>11} {'$ cached':>9}")
for n in (1, 5, 10, 15, 30):
    t, c = loop_cost(n); _, cc = loop_cost(n, cached=True)
    print(f"{n:>5} {t:>12,} {c:>11.3f} {cc:>9.3f}")

```code
steps  tokens sent  $ uncached  $ cached
    1        1,550       0.003     0.004
    5       23,550       0.047     0.023
   10       86,600       0.173     0.054
   15      189,150       0.378     0.092
   30      733,800       1.468     0.256
```

## 5. Loop hygiene

1. Shrink at the source (tool params). 2. Hard cap every result. 3. **Say it was cut and how to get the rest.** 4. Keep pointers (line numbers, ids).

In [ ]:
LOG = [f"{i:04d}\t09:{i // 60:02d}:{i % 60:02d}\t{'ERROR' if i % 7 == 0 else 'INFO'}\tcheckout\tmsg {i}"
       for i in range(3000)]                                   # fake 3,000-line incident log, line numbers kept

def read_logs(level=None, pattern=None, limit=50, max_chars=8000):
    hits = [l for l in LOG if (level is None or f"\t{level}\t" in l) and (pattern is None or pattern in l)]
    if not hits:
        return f"0 lines matched level={level} pattern={pattern}. Try a wider filter."   # an answer, not is_error
    shown = hits[:limit]
    body = "\n".join(shown)[:max_chars]
    if len(shown) < len(hits) or len(body) < len("\n".join(shown)):
        body += f"\n[Showing {len(shown)} of {len(hits)} matching lines. Narrow with pattern= or raise limit.]"
    return body

full = "\n".join(LOG)
capped = read_logs(level="ERROR")
print(f"raw: {len(full):,} chars (~{len(full)//4:,} tokens)   capped: {len(capped):,} chars (~{len(capped)//4:,} tokens)")
print(capped.splitlines()[-1])
print(read_logs(pattern="no-such-text"))

```code
raw: 110,318 chars (~27,579 tokens)   capped: 1,905 chars (~476 tokens)
[Showing 50 of 429 matching lines. Narrow with pattern= or raise limit.]
0 lines matched level=None pattern=no-such-text. Try a wider filter.
```

### 5.1 One-line trace + per-run transcript
Same events feed both: stderr for watching live, `runs/<run_id>.jsonl` for later. Append after **every** event so a crash keeps steps so far.

In [ ]:
import sys
RUNS = Path("runs"); RUNS.mkdir(exist_ok=True)

def make_logger(run_id: str):
    path = RUNS / f"{run_id}.jsonl"
    def log(event: dict):
        event = {"ts": round(time.time(), 3), "run_id": run_id, **event}
        with path.open("a") as f:
            f.write(json.dumps(event, default=str) + "\n")    # append, one event per line
        if event["type"] == "llm":                            # one-line trace
            u = event["usage"]; calls = ", ".join(f"{t['name']}({json.dumps(t['input'])})" for t in event["tool_uses"])
            print(f"[{run_id}] s{event['step']:02d} {event['stop_reason']:<9} {calls or '-'} "
                  f"| in {u['in']} (cache r{u['cache_read']}/w{u['cache_write']}) out {u['out']}", file=sys.stderr)
        elif event["type"] == "tool":
            flag = "ERROR" if event["is_error"] else "ok"
            print(f"[{run_id}]     -> {event['name']} {flag} {len(event['sent'])} chars", file=sys.stderr)
    return log, path

def run_logged(cli, task, **kw):
    run = Run(**kw); run_id = "r_" + uuid.uuid4().hex[:6]
    run.log, path = make_logger(run_id)
    run.log({"type": "run_start", "model": MODEL, "task": task,
             "tools_sha": __import__("hashlib").sha256(json.dumps(TOOLS).encode()).hexdigest()[:8]})
    msgs, outcome = agent_loop(cli, task, run)
    run.log({"type": "run_end", "outcome": outcome, "steps": run.steps, "input_tokens": run.used_input})
    return msgs, outcome, path

# offline demo with the fake client
fc = FakeClient([
    fake_resp(tu("A", "get_order", order_id="1042"), tu("B", "check_stock", sku="blue mug")),
    fake_resp(tu("C", "check_stock", sku="MUG-BLU"), inp=3200),
    fake_resp(tx("Shipped; blue mug out of stock."), stop="end_turn", inp=3400),
])
_, outcome, path = run_logged(fc, "Order 1042 and blue mug?")
print(outcome, path)

```code
end_turn runs/r_494ab3.jsonl
[r_494ab3] s01 tool_use  get_order({"order_id": "1042"}), check_stock({"sku": "blue mug"}) | in 1500 (cache r0/w0) out 80
[r_494ab3]     -> get_order ok 23 chars
[r_494ab3]     -> check_stock ERROR 73 chars
[r_494ab3] s02 tool_use  check_stock({"sku": "MUG-BLU"}) | in 3200 (cache r0/w0) out 80
[r_494ab3]     -> check_stock ok 19 chars
[r_494ab3] s03 end_turn  - | in 3400 (cache r0/w0) out 80
```

### 5.2 Transcript viewer
What the plan means by "you can explain every message in one transcript".

In [ ]:
def show_transcript(path: Path):
    for line in path.read_text().splitlines():
        e = json.loads(line)
        if e["type"] == "run_start":
            print(f"RUN {e['run_id']} model={e['model']} tools_sha={e['tools_sha']}\n  task: {e['task']}")
        elif e["type"] == "llm":
            print(f"step {e['step']}: model -> {e['stop_reason']}  {e['tool_uses'] or ''}  usage={e['usage']}")
        elif e["type"] == "tool":
            print(f"   tool {e['name']}({e['input']}) {'ERROR ' if e['is_error'] else ''}-> {e['sent'][:70]!r}")
        elif e["type"] == "run_end":
            print(f"END outcome={e['outcome']} steps={e['steps']} input_tokens={e['input_tokens']}")

show_transcript(path)

```code
RUN r_494ab3 model=claude-haiku-5-5 tools_sha=a2e1cae8
  task: Order 1042 and blue mug?
step 1: model -> tool_use  [{'name': 'get_order', 'input': {'order_id': '1042'}}, {'name': 'check_stock', 'input': {'sku': 'blue mug'}}]  usage={'in': 1500, 'out': 80, 'cache_read': 0, 'cache_write': 0}
   tool get_order({'order_id': '1042'}) -> 'shipped, arrives Nov 11'
   tool check_stock({'sku': 'blue mug'}) ERROR -> "ValueError: Unknown sku 'blue mug'. Valid skus: MUG-BLU, MUG-RED, TEE-"
step 2: model -> tool_use  [{'name': 'check_stock', 'input': {'sku': 'MUG-BLU'}}]  usage={'in': 3200, 'out': 80, 'cache_read': 0, 'cache_write': 0}
   tool check_stock({'sku': 'MUG-BLU'}) -> 'MUG-BLU: 0 in stock'
step 3: model -> end_turn    usage={'in': 3400, 'out': 80, 'cache_read': 0, 'cache_write': 0}
END outcome=end_turn steps=3 input_tokens=8100
```

### 5.3 Real logged run  [$]

In [ ]:
_, outcome, path = run_logged(client, "Is the blue mug in stock? If not, what colour is?", max_steps=6)
show_transcript(path)

```code
[r_bdfb1b] s01 tool_use  check_stock({"sku": "MUG-BLUE"}) | in 4 (cache r0/w533) out 175
[r_bdfb1b]     -> check_stock ERROR 73 chars
[r_bdfb1b] s02 tool_use  check_stock({"sku": "MUG-BLU"}), check_stock({"sku": "MUG-RED"}) | in 2 (cache r533/w239) out 309
[r_bdfb1b]     -> check_stock ok 19 chars
[r_bdfb1b]     -> check_stock ok 19 chars
RUN r_bdfb1b model=claude-haiku-5-5 tools_sha=a2e1cae8
  task: Is the blue mug in stock? If not, what colour is?
step 1: model -> tool_use  [{'name': 'check_stock', 'input': {'sku': 'MUG-BLUE'}}]  usage={'in': 4, 'out': 175, 'cache_read': 0, 'cache_write': 533}
   tool check_stock({'sku': 'MUG-BLUE'}) ERROR -> "ValueError: Unknown sku 'MUG-BLUE'. Valid skus: MUG-BLU, MUG-RED, TEE-"
step 2: model -> tool_use  [{'name': 'check_stock', 'input': {'sku': 'MUG-BLU'}}, {'name': 'check_stock', 'input': {'sku': 'MUG-RED'}}]  usage={'in': 2, 'out': 309, 'cache_read': 533, 'cache_write': 239}
   tool check_stock({'sku': 'MUG-BLU'}) -> 'MUG-BLU: 0 in stock'
   tool check_stock({'sku': 'MUG-RED'}) -> 'MUG-RED: 7 in stock'
step 3: model -> end_turn    usage={'in': 2, 'out': 32, 'cache_read': 772, 'cache_write': 394}
END outcome=end_turn steps=3 input_tokens=2479
[r_bdfb1b] s03 end_turn  - | in 2 (cache r772/w394) out 32
```

#### END